# 07 · Guardrails: Validating Task Outputs

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama.

An `expected_output` *asks* for a certain result; a **guardrail checks** that the
result meets your rules, and if it doesn't, sends the agent back to try again with
feedback. CrewAI guardrails sit on **tasks**: they validate each task's output
before it's accepted and passed on. This notebook covers code guardrails, LLM
guardrails, and what happens when an agent simply can't comply.

**You will learn**
- Function guardrails: `(ok, value_or_feedback)`, and automatic retries with feedback
- Guardrails that **fix** the output instead of rejecting it
- What happens when retries run out, and the **persona that fights its guardrail**
- **LLM guardrails**: rules written in plain English, judged by a model
- Chaining several guardrails, and combining them with structured output

**Prerequisites:** notebooks 02–03.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import re
from crewai import Agent, Task, Crew, TaskOutput

llm = get_llm()

## 1 · Why guardrails

The model is asked for "a tagline of at most 20 words" and returns 45. Without a
guardrail, those 45 words flow into the next task, or to a customer. With one:

```
 agent's output ──► guardrail(output)
                        │
          (True, value) ├──► accepted: becomes the task's output
                        │
  (False, "feedback")   └──► the agent tries again, with the feedback added
                             … up to guardrail_max_retries times, then the task FAILS
```

A guardrail is a function `guardrail(output: TaskOutput) -> (bool, value)`:

- `(True, value)`: accept. `value` becomes the task's output, so you can also clean it up.
- `(False, "message")`: reject. The message is sent to the agent as feedback for its retry.

## 2 · A code guardrail with retries

In [ ]:
attempts = []

def at_most_20_words(output: TaskOutput):
    words = len(output.raw.split())
    attempts.append(words)
    if words <= 20:
        return True, output.raw
    return False, f'Too long: {words} words. Rewrite it in at most 20 words.'


copywriter = Agent(role='Copywriter', goal='Write punchy taglines',
                   backstory='You write short, concrete copy.', llm=llm)
tagline = Task(description='Write a tagline for sidewalk delivery robots.', expected_output='One tagline',
               agent=copywriter, guardrail=at_most_20_words, guardrail_max_retries=3)

out = await Crew(agents=[copywriter], tasks=[tagline]).kickoff_async()
print('word counts per attempt:', attempts)
print('accepted:', out.raw)

The guardrail runs on every attempt; `attempts` shows the word count each time.
If an attempt fails, the agent gets the feedback message and tries again.

## 3 · Guardrails can fix, not just reject

Because an accepting guardrail returns the value to use, it can **normalise** the
output: strip quotes and markdown, fix formatting, redact something. It's
cheaper than sending the agent back for a cosmetic problem:

In [ ]:
def clean_tagline(output: TaskOutput):
    text = output.raw.replace('**', '').strip().strip('"').strip("'").strip()   # remove bold FIRST, then quotes
    if not text:
        return False, 'The tagline is empty. Write one tagline.'
    return True, text                                  # the CLEANED text becomes the task output

tidy = Task(description='Write a tagline for sidewalk delivery robots, in quotes and bold.',
            expected_output='One tagline', agent=copywriter, guardrail=clean_tagline)
out = await Crew(agents=[copywriter], tasks=[tidy]).kickoff_async()
print(repr(out.raw))

## 4 · When the agent can't comply: the persona fights the guardrail

Now the same guardrail with a different agent: a marketer whose backstory says
they write long, enthusiastic paragraphs. The task asks for 20 words; the
persona pulls the other way:

In [ ]:
attempts.clear()
marketer = Agent(role='Marketer', goal='Excite customers',
                 backstory='You are a wordy marketer who writes long, enthusiastic paragraphs.', llm=llm)
long_tagline = Task(description='Write a tagline for sidewalk delivery robots.', expected_output='One tagline',
                    agent=marketer, guardrail=at_most_20_words, guardrail_max_retries=3)
try:
    await Crew(agents=[marketer], tasks=[long_tagline]).kickoff_async()
except Exception as err:
    print('word counts per attempt:', attempts)
    print(type(err).__name__, '→', str(err)[:160])

In our runs the marketer failed **every** attempt (e.g. 85, 24, 69, 52 words):
the feedback pulled it toward shorter text once, then the persona won again. When
retries run out, the task **fails with a plain `Exception`** (not a specific
exception type), so the entry point that runs the crew must catch it.

Two lessons:

1. **Guardrails can't fix a design conflict.** If the persona contradicts the rule,
   change the persona, not the retry count. The concise copywriter in §2 passed
   (usually on the first try).
2. **Plan for the failure path.** Catch the exception and decide what happens next:
   a fallback text, a human, or an honest error.

## 5 · LLM guardrails: rules in plain English

Some rules are hard to code ("is this free of hype?", "does it answer the
question?"). Give `guardrail=` a **string** instead of a function, and CrewAI
asks a model to judge the output against it (using the agent's LLM by default).
If the judge says no, its explanation becomes the retry feedback:

In [ ]:
hype_rule = 'The text must not contain exclamation marks, superlatives or unverifiable claims.'

sober = Task(description='Write a two-sentence product description for sidewalk delivery robots.',
             expected_output='Two sentences', agent=marketer, guardrail=hype_rule, guardrail_max_retries=3)
try:
    out = await Crew(agents=[marketer], tasks=[sober]).kickoff_async()
    print('accepted:', out.raw)
except Exception as err:
    print('failed:', str(err)[:200])

LLM guardrails are convenient, but remember what they are: **another model call
per attempt** (cost and latency), and **a model making the judgement**, which can
be wrong or inconsistent. Prefer code for anything you can check exactly
(lengths, formats, required words, forbidden patterns), and use LLM guardrails
for genuine judgement calls.

## 6 · Several guardrails, and structured output

`guardrails=[g1, g2, …]` runs them **in order**, each receiving the previous
one's accepted value, so you can chain *clean → check → check*. Code and string
guardrails can be mixed.

Guardrails also work with `output_pydantic`, which is the best way to enforce
**business rules across fields** that a schema alone can't express:

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Ticket(BaseModel):
    category: Literal['billing', 'technical', 'other']
    urgency: int = Field(ge=1, le=5)
    escalate: bool

def consistent(output: TaskOutput):
    t = output.pydantic
    if t.urgency >= 4 and not t.escalate:
        return False, f'Urgency {t.urgency} must be escalated. Set escalate=true.'
    return True, output

def no_money_words_in_technical(output: TaskOutput):
    return True, output                                # a placeholder for a second rule, run after `consistent`

triager = Agent(role='Support triage', goal='Classify tickets', backstory='Precise.', llm=llm)
classify = Task(description=('Classify this ticket. Urgency 1-5 (5 = losing money now or deadline within 48h). '
                             'Ticket: {ticket}'),
                expected_output='A classified ticket', agent=triager, output_pydantic=Ticket,
                guardrails=[consistent, no_money_words_in_technical], guardrail_max_retries=2)
out = await Crew(agents=[triager], tasks=[classify]).kickoff_async(
    inputs={'ticket': 'Charged three times, rent due tomorrow!!'})
print(out.pydantic)

The schema guarantees each field's type and range; the guardrail guarantees they
**agree with each other** (high urgency ⇒ escalate).

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| A persona that contradicts the rule | every retry fails | fix the persona or the task, not the retry count |
| Not catching the final failure | a crash with a generic `Exception` | catch it at the entry point; decide the fallback |
| Vague feedback messages | retries make the same mistake | say exactly what's wrong and what to do |
| LLM guardrails for exact checks | extra cost, inconsistent verdicts | code for lengths, formats, patterns |
| Rejecting for cosmetic issues | wasted retries | return `(True, cleaned_value)` |

## 8 · Check yourself

<details><summary><b>1.</b> What does a guardrail function return, and what happens in each case?</summary>

`(True, value)`: accepted, and `value` becomes the task output. `(False, message)`: rejected, and the agent retries with `message` as feedback, up to `guardrail_max_retries` times.
</details>

<details><summary><b>2.</b> The retries run out. What happens, and what should your code do?</summary>

The task fails with a plain `Exception`. Catch it where you run the crew and choose a fallback: a default output, a human review, or an honest error.
</details>

<details><summary><b>3.</b> Why did the marketer keep failing the 20-word rule?</summary>

Its backstory (wordy, long enthusiastic paragraphs) is part of its system prompt and pulls against the rule. Retries can't resolve a contradiction built into the agent.
</details>

<details><summary><b>4.</b> When would you use a string (LLM) guardrail instead of a function?</summary>

For judgement calls that code can't check exactly (tone, relevance, hype), accepting the extra model call and the possibility of a wrong verdict.
</details>

## Takeaway

Task guardrails turn "please follow these rules" into **check and retry with
feedback**. Use code for exact rules and LLM guardrails for judgement, let
guardrails clean up cosmetic issues, chain them for several rules, and always
handle the final failure, which arrives as a plain `Exception`. No guardrail can
fix an agent whose persona contradicts the task.

Next → `08_human_in_the_loop.ipynb`: pausing a flow for a human decision.